## nb_03_1_player_info_silver

Cleans and validates table in bronze schema and and writes the result to sliver schema
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

StatementMeta(, 86f0b96f-2ce3-469b-8bff-fe981f8efb5e, 3, Finished, Available, Finished, False)

### Load common db functions

In [2]:
%run nb_00_dbutils

StatementMeta(, 86f0b96f-2ce3-469b-8bff-fe981f8efb5e, 15, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded:

```
%run <notebook name> { "RUN_PIPELINE": false }
```

In [3]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 86f0b96f-2ce3-469b-8bff-fe981f8efb5e, 16, Finished, Available, Finished, False)

### Config

In [4]:
BRONZE_TABLE: str = "bronze.player_info"
SILVER_TABLE: str = "silver.player_info"

# Only these columns make it into the silver table
required_cols: list[str] = [
    "player_id",
    "firstName",
    "lastName",
    "birthDate",
    "primaryPosition",
    "nationality",
]

fill_null_cols: dict[str, str] = {
    "nationality": "None"
}

# Natural key used to de-duplicate rows
PRIMARY_KEYS: list[str] = ["player_id"]
DEDUPE_KEYS: list[str] = ["player_id"]

# Valid player position values as tuples
VALID_PRIMARY_POSITION: tuple[str] = ('LW', 'D', 'C', 'RW', 'G')


StatementMeta(, 86f0b96f-2ce3-469b-8bff-fe981f8efb5e, 17, Finished, Available, Finished, False)

## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

In [5]:
if RUN_PIPELINE:
    
    # only load latest dataset to existing silver table
    df: DataFrame = (
        read_new_bronze_rows(
            spark, 
            BRONZE_TABLE, 
            SILVER_TABLE, 
            columns=required_cols
        )
    )

    if not df.isEmpty():
        # ensure latest data is kept 
        # no duplicates based on primary keys
        df = keep_latest_per_key(df, PRIMARY_KEYS)

        # cleaning and transforms
        df = df.dropDuplicates(DEDUPE_KEYS)
        df = df.fillna(fill_null_cols)
        df = df.withColumn(
            "birthdate",
            F.to_date(F.col("birthdate"))
        )

        # Data quality checks
        df = validate_no_nulls(df, columns=required_cols)
        df = validate_column_values(df, column="primaryPosition", values=VALID_PRIMARY_POSITION)

        df = validate_primary_keys(df, keys=PRIMARY_KEYS)

        # write to SCD Type 1 silver table with latest records only
        write_to_silver(spark, df, SILVER_TABLE, PRIMARY_KEYS) 
        print(f"🏁 {SILVER_TABLE} load complete.")
    
    else:
        print(f"No new data for {SILVER_TABLE}")

StatementMeta(, 86f0b96f-2ce3-469b-8bff-fe981f8efb5e, 18, Finished, Available, Finished, False)

[bronze.player_info] Reading rows after watermark 2026-09-28 01:55:21.260282
[silver.player_info] Merge complete.
🏁 silver.player_info load complete.
